# Objective metrics

An **objective metric** is the single number you optimize and compare models by. The
right choice depends on the problem type. This notebook covers the 3 most useful
metrics for each of the two most common problems:

- **Binary (binomial) classification** - ROC AUC, F1, Log loss
- **Regression** - RMSE, MAE, R^2

For each metric: what it measures, its range, and what counts as a **good** vs **bad**
value - with a tiny runnable `scikit-learn` example.

## At a glance

| Metric | Problem | Needs | Range | Better | Best | No-skill baseline |
|--------|---------|-------|-------|--------|------|-------------------|
| **ROC AUC** | binary | probabilities | 0.0 - 1.0 | higher | 1.0 | 0.5 (random) |
| **F1** | binary | hard 0/1 labels | 0.0 - 1.0 | higher | 1.0 | 0 if precision or recall is 0 |
| **Log loss** | binary | probabilities | 0.0 - +inf | lower | 0.0 | ~0.693 (ln 2, on balanced data) |
| **RMSE** | regression | predictions | 0.0 - +inf | lower | 0.0 | target std (predicting the mean) |
| **MAE** | regression | predictions | 0.0 - +inf | lower | 0.0 | mean abs deviation from the mean |
| **R^2** | regression | predictions | -inf - 1.0 | higher | 1.0 | 0.0 (predicting the mean); < 0 is worse |

Each metric is detailed below with a runnable example.

## Setup

`.venv-notebooks` is a `uv`-managed venv (no `pip` inside it), so we install with
`uv pip install` pointed at this kernel's interpreter.

In [ ]:
import sys

!uv pip install --python "{sys.executable}" scikit-learn numpy

In [ ]:
import numpy as np
import sklearn.metrics as m

print("scikit-learn ready")

# Binary (binomial) classification metrics

Shared toy example: the true labels, the model's predicted **probabilities** of the
positive class, and the **hard 0/1 predictions** (threshold 0.5) derived from them.
Some metrics need probabilities (ROC AUC, Log loss), others need hard labels (F1).

In [ ]:
# true labels (1 = positive)
y_true = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1])

# model's predicted probability of the positive class
y_prob = np.array([0.1, 0.2, 0.35, 0.6, 0.4, 0.65, 0.7, 0.8, 0.9, 0.95])

# hard predictions at the usual 0.5 threshold
y_pred = (y_prob >= 0.5).astype(int)

print("y_true:", y_true)
print("y_pred:", y_pred)

## 1. ROC AUC (`roc_auc_score`)

Probability that a randomly chosen **positive** is ranked above a randomly chosen
**negative**. Uses predicted probabilities/scores, so it is **threshold-independent**
and robust to class imbalance in the ranking sense.

| Range | Good | Bad |
|-------|------|-----|
| 0.0 - 1.0 (higher better) | **1.0** perfect ranking; **>= 0.9** excellent; **~0.8** strong | **0.5** = random / no skill (useless); **< 0.5** worse than random (ranking inverted) |

In [ ]:
print("ROC AUC:", round(m.roc_auc_score(y_true, y_prob), 3))

## 2. F1 score (`f1_score`)

Harmonic mean of **precision** (of predicted positives, how many are right) and
**recall** (of actual positives, how many are caught). Needs hard labels. Balances
false positives against false negatives - far more informative than accuracy when
classes are imbalanced.

| Range | Good | Bad |
|-------|------|-----|
| 0.0 - 1.0 (higher better) | **1.0** perfect precision *and* recall; close to 1 | **0.0** when precision or recall collapses to 0; low values mean many FPs and/or missed positives |

In [ ]:
print("Precision:", round(m.precision_score(y_true, y_pred), 3))
print("Recall:   ", round(m.recall_score(y_true, y_pred), 3))
print("F1:       ", round(m.f1_score(y_true, y_pred), 3))

## 3. Log loss / binary cross-entropy (`log_loss`)

Penalizes predicted probabilities by how far and how **confidently** they are wrong.
A confident wrong prediction is punished heavily. Rewards well-**calibrated**
probabilities, not just correct ranking.

| Range | Good | Bad |
|-------|------|-----|
| 0.0 - +inf (lower better) | **0.0** perfect, confident, correct; close to 0 | **~0.693** (= ln 2) is what 50/50 guessing scores on a balanced set - anything near/above that is no skill; large values = confidently wrong |

In [ ]:
print("Log loss:        ", round(m.log_loss(y_true, y_prob), 3))
print("ln(2) reference: ", round(float(np.log(2)), 3), "(random guessing)")

# Regression metrics

Shared toy example: true continuous targets and the model's predictions. RMSE and MAE
are in the **same units as the target**, so judge them relative to the target's scale
(e.g. against its standard deviation or mean); R^2 is unit-free.

In [ ]:
y_true_r = np.array([10.0, 12.0, 14.0, 16.0, 18.0, 20.0])
y_pred_r = np.array([10.5, 11.0, 14.5, 15.0, 19.0, 21.5])

print("target mean:", y_true_r.mean(), "  target std:", round(float(y_true_r.std()), 3))

## 1. RMSE - root mean squared error

Square root of the mean squared error. Same units as the target. Because errors are
squared first, **large errors dominate** - use it when big misses are especially bad.

| Range | Good | Bad |
|-------|------|-----|
| 0.0 - +inf (lower better) | **0.0** perfect; small **relative to the target's scale** (e.g. << target std) | large relative to the target's scale; a single big miss can blow it up |

In [ ]:
rmse = np.sqrt(m.mean_squared_error(y_true_r, y_pred_r))
print("RMSE:", round(float(rmse), 3))

## 2. MAE - mean absolute error

Average absolute difference between prediction and truth. Same units as the target.
Every error counts linearly, so it is **more robust to outliers** than RMSE and reads
as a plain "typical error of X units".

| Range | Good | Bad |
|-------|------|-----|
| 0.0 - +inf (lower better) | **0.0** perfect; small relative to the target's scale | large relative to the target's scale |

In [ ]:
print("MAE:", round(float(m.mean_absolute_error(y_true_r, y_pred_r)), 3))

## 3. R^2 - coefficient of determination (`r2_score`)

Fraction of the target's variance the model explains. Unit-free, so it is comparable
across datasets. The baseline it compares against is "always predict the mean".

| Range | Good | Bad |
|-------|------|-----|
| -inf - 1.0 (higher better) | **1.0** perfect; close to 1 explains most variance (>0.7 often decent, domain-dependent) | **0.0** = no better than predicting the mean; **< 0** = worse than predicting the mean |

In [ ]:
print("R^2:", round(float(m.r2_score(y_true_r, y_pred_r)), 3))